# Progressive Reveal

K-space is acquired from the center outward. The center encodes coarse structure; the edges encode fine detail.

This notebook visualizes what the knee looks like at each stage of acquisition — from a blurry mass of tissue to full resolution. It's both a diagnostic tool (how much data do you really need?) and a patient-education tool (show someone their scan assembling itself).

It's also the principle behind **compressed sensing** — how modern scanners reconstruct full images from partial k-space data.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from kode.io import load_kspace
from kode.progressive_reveal import progressive_frames

In [ ]:
H5_FILE = '../data/your_file.h5'
kspace = load_kspace(H5_FILE, slice_idx=15)
print(f'K-space shape: {kspace.shape}')

frames = progressive_frames(kspace, steps=8)
print(f'Generated {len(frames)} frames')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
percentages = [f'{int((i+1)/8*100)}% of k-space' for i in range(8)]

for ax, frame, label in zip(axes.flat, frames, percentages):
    ax.imshow(frame, cmap='gray')
    ax.set_title(label, fontsize=10)
    ax.axis('off')

plt.suptitle('Progressive Reveal — From DC Component to Full Resolution', fontsize=13)
plt.tight_layout()
plt.savefig('../results/progressive_reveal.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save as animated GIF — shows the scan assembling itself
try:
    from PIL import Image

    gif_frames = []
    for frame in frames:
        normalized = ((frame / frame.max()) * 255).astype('uint8')
        gif_frames.append(Image.fromarray(normalized))

    gif_frames[0].save(
        '../results/progressive_reveal.gif',
        save_all=True,
        append_images=gif_frames[1:],
        duration=300,
        loop=0,
    )
    print('GIF saved to results/progressive_reveal.gif')
except ImportError:
    print('Pillow not installed — run: pip install Pillow')